# Pre-LLM Survey Validation

This notebook evaluates the pre-LLM survey stage on two fixed cohorts:
- 25 names with known Purdue relationships
- 25 names intended to be strongly unrelated to Purdue

The goal is to judge whether the survey buckets names as expected before any LLM call:
- Purdue-related names should usually avoid `hard_no`
- super non-related names should usually land in `hard_no`


In [148]:
import asyncio
import importlib
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import institution_checker.search as ic_search
ic_search = importlib.reload(ic_search)

import institution_checker.main as ic_main
ic_main = importlib.reload(ic_main)

from institution_checker import INSTITUTION, close_search_clients, close_session

print(f"Repo root: {repo_root}")
print(f"Institution: {INSTITUTION}")


Repo root: e:\Awards DB Code\step3_institution_checker
Institution: Purdue University


In [149]:
PURDUE_NAMES = [
    "Neil Armstrong",
    "Drew Brees",
    "Amelia Earhart",
    "Mitch Daniels",
    "France A. Cordova",
    "Mung Chiang",
    "Eugene H. Spafford",
    "Ei-ichi Negishi",
    "Herbert C. Brown",
    "R. Graham Cooks",
    "John B. Fenn",
    "Vernon L. Smith",
    "Gus Grissom",
    "Chesley Sullenberger",
    "Brian Lamb",
    "Leah Jamieson",
    "Michael G. Rossmann",
    "Samuel D. Allen",
    "Edward M. Purcell",
    "Ted Allen",
    "Raj Rajamani",
    "Marisol Nunez",
    "Katherine Banks",
    "Timothy D. Sands",
    "Jozef L. Kokini",
]

NON_PURDUE_NAMES = [
    "Taylor Swift",
    "Lionel Messi",
    "Beyonce Knowles",
    "Adele Adkins",
    "Serena Williams",
    "Roger Federer",
    "Usain Bolt",
    "Frida Kahlo",
    "Pablo Picasso",
    "David Attenborough",
    "Cristiano Ronaldo",
    "Meryl Streep",
    "Keanu Reeves",
    "Tom Hanks",
    "Oprah Winfrey",
    "Billie Eilish",
    "Zendaya",
    "Rihanna",
    "Shakira",
    "Emma Stone",
    "Robert Downey Jr",
    "Simone Biles",
    "Stephen Curry",
    "LeBron James",
    "Celine Dion",
]

assert len(PURDUE_NAMES) == 25
assert len(NON_PURDUE_NAMES) == 25

cohort_df = pd.DataFrame(
    [{"name": name, "expected_group": "purdue_related", "expected_bucket_ok": "not_hard_no"} for name in PURDUE_NAMES]
    + [{"name": name, "expected_group": "super_non_related", "expected_bucket_ok": "hard_no"} for name in NON_PURDUE_NAMES]
)
cohort_df


,name,expected_group,expected_bucket_ok
0,Neil Armstrong,purdue_related,not_hard_no
1,Drew Brees,purdue_related,not_hard_no
2,Amelia Earhart,purdue_related,not_hard_no
3,Mitch Daniels,purdue_related,not_hard_no
4,France A. Cordova,purdue_related,not_hard_no
5,Mung Chiang,purdue_related,not_hard_no
6,Eugene H. Spafford,purdue_related,not_hard_no
7,Ei-ichi Negishi,purdue_related,not_hard_no
8,Herbert C. Brown,purdue_related,not_hard_no
9,R. Graham Cooks,purdue_related,not_hard_no


In [150]:
# Runtime controls
USE_ENHANCED_SEARCH = True
VALIDATION_POLICY = "fast_triage"
ALLOW_BING_FALLBACK = False
ALLOW_SLOW_DDG_FALLBACK = False
ENABLE_NOTEBOOK_CACHE = True
DATASET_PROFILE = "high_connection"
DEBUG = False
SAVE_RESULTS = True
CONCURRENT_SEARCHES = 10
OUTPUT_PATH = repo_root / "data" / "pre_llm_survey_validation_results.csv"

# Optional: uncomment if your notebook environment does not already have the API key.
# from institution_checker import set_api_key
# set_api_key("...")

print({
    "USE_ENHANCED_SEARCH": USE_ENHANCED_SEARCH,
    "VALIDATION_POLICY": VALIDATION_POLICY,
    "DATASET_PROFILE": DATASET_PROFILE,
    "ALLOW_BING_FALLBACK": ALLOW_BING_FALLBACK,
    "ALLOW_SLOW_DDG_FALLBACK": ALLOW_SLOW_DDG_FALLBACK,
    "ENABLE_NOTEBOOK_CACHE": ENABLE_NOTEBOOK_CACHE,
    "DEBUG": DEBUG,
    "CONCURRENT_SEARCHES": CONCURRENT_SEARCHES,
    "OUTPUT_PATH": str(OUTPUT_PATH),
})


{'USE_ENHANCED_SEARCH': True, 'VALIDATION_POLICY': 'fast_triage', 'DATASET_PROFILE': 'high_connection', 'ALLOW_BING_FALLBACK': False, 'ALLOW_SLOW_DDG_FALLBACK': False, 'ENABLE_NOTEBOOK_CACHE': True, 'DEBUG': False, 'CONCURRENT_SEARCHES': 10, 'OUTPUT_PATH': 'e:\\Awards DB Code\\step3_institution_checker\\data\\pre_llm_survey_validation_results.csv'}


## Run Search + Pre-LLM Survey

This runs actual search for each name, then applies the survey stage only. It does not call the LLM.

In [151]:
import time

async def survey_name(name: str, expected_group: str) -> dict:
    start = time.perf_counter()
    if VALIDATION_POLICY == "fast_triage":
        surveyed_results, decision, validation_meta = await ic_main.run_pre_llm_validation_search(
            name,
            dataset_profile=DATASET_PROFILE,
            debug=DEBUG,
            allow_enhanced=USE_ENHANCED_SEARCH,
            allow_bing_fallback=ALLOW_BING_FALLBACK,
            allow_slow_ddg_fallback=ALLOW_SLOW_DDG_FALLBACK,
            cache_enabled=ENABLE_NOTEBOOK_CACHE,
        )
        search_results = validation_meta.get("basic_result_count", len(surveyed_results))
        search_mode_used = validation_meta.get("search_mode_used", "basic_only")
        enhanced_escalated = bool(validation_meta.get("enhanced_escalated", False))
        rescue_attempted = bool(validation_meta.get("rescue_attempted", False))
        basic_result_count = validation_meta.get("basic_result_count", len(surveyed_results))
        final_result_count = validation_meta.get("final_result_count", len(surveyed_results))
        escalation_reason = validation_meta.get("escalation_reason", "")
        rescue_reason = validation_meta.get("rescue_reason", "")
        enhanced_reason = validation_meta.get("enhanced_reason", "")
        borderline_subtype = validation_meta.get("borderline_subtype", "")
        backend_used = validation_meta.get("backend_used", "")
        cache_hit = bool(validation_meta.get("cache_hit", False))
        network_queries_used = validation_meta.get("network_queries_used", 0)
        cache_hits_total = validation_meta.get("cache_hits_total", 0)
        cache_misses_total = validation_meta.get("cache_misses_total", 0)
        ddg_manual_retry_used = bool(validation_meta.get("ddg_manual_retry_used", False))
        ddg_browser_fallback_used = bool(validation_meta.get("ddg_browser_fallback_used", False))
        bing_fallback_used = bool(validation_meta.get("bing_fallback_used", False))
        network_attempt_count = validation_meta.get("network_attempt_count", 0)
    else:
        _, results = await ic_main.process_name_search(
            name,
            use_enhanced_search=USE_ENHANCED_SEARCH,
            debug=DEBUG,
            dataset_profile=DATASET_PROFILE,
        )
        surveyed_results, decision = await ic_main.run_pre_llm_survey(
            name,
            results,
            dataset_profile=DATASET_PROFILE,
            debug=DEBUG,
        )
        search_results = len(results)
        search_mode_used = "production_path"
        enhanced_escalated = bool(USE_ENHANCED_SEARCH)
        rescue_attempted = bool(decision.used_rescue_query)
        basic_result_count = len(results)
        final_result_count = len(surveyed_results)
        escalation_reason = ""
        rescue_reason = ""
        enhanced_reason = ""
        borderline_subtype = ""
        backend_used = "production_path"
        cache_hit = False
        network_queries_used = None
        cache_hits_total = None
        cache_misses_total = None
        ddg_manual_retry_used = None
        ddg_browser_fallback_used = None
        bing_fallback_used = None
        network_attempt_count = None

    elapsed_seconds = round(time.perf_counter() - start, 2)

    metrics = dict(decision.metrics)
    expected_pass = expected_group == "purdue_related"
    actual_pass = decision.bucket != ic_main.SURVEY_HARD_NO
    strict_expected = (decision.bucket != ic_main.SURVEY_HARD_NO) if expected_pass else (decision.bucket == ic_main.SURVEY_HARD_NO)

    return {
        "name": name,
        "expected_group": expected_group,
        "search_results": search_results,
        "survey_results": len(surveyed_results),
        "search_mode_used": search_mode_used,
        "enhanced_escalated": enhanced_escalated,
        "rescue_attempted": rescue_attempted,
        "basic_result_count": basic_result_count,
        "final_result_count": final_result_count,
        "elapsed_seconds": elapsed_seconds,
        "escalation_reason": escalation_reason,
        "rescue_reason": rescue_reason,
        "enhanced_reason": enhanced_reason,
        "borderline_subtype": borderline_subtype,
        "backend_used": backend_used,
        "cache_hit": cache_hit,
        "network_queries_used": network_queries_used,
        "ddg_manual_retry_used": ddg_manual_retry_used,
        "ddg_browser_fallback_used": ddg_browser_fallback_used,
        "bing_fallback_used": bing_fallback_used,
        "network_attempt_count": network_attempt_count,
        "cache_hits_total": cache_hits_total,
        "cache_misses_total": cache_misses_total,
        "bucket": decision.bucket,
        "score": decision.score,
        "summary": decision.summary,
        "reason_codes": "|".join(decision.reason_codes),
        "used_rescue_query": decision.used_rescue_query,
        "stage": metrics.get("stage", ""),
        "max_relevance_score": metrics.get("max_relevance_score", 0),
        "purdue_mentions": metrics.get("purdue_mentions", 0),
        "purdue_domain_hits": metrics.get("purdue_domain_hits", 0),
        "edu_hits": metrics.get("edu_hits", 0),
        "explicit_connection_hits": metrics.get("explicit_connection_hits", 0),
        "authoritative_institution_hits": metrics.get("authoritative_institution_hits", 0),
        "person_institution_hits": metrics.get("person_institution_hits", 0),
        "academic_role_hits": metrics.get("academic_role_hits", 0),
        "degree_hits": metrics.get("degree_hits", 0),
        "joint_campus_hits": metrics.get("joint_campus_hits", 0),
        "negative_keyword_hits": metrics.get("negative_keyword_hits", 0),
        "negative_result_hits": metrics.get("negative_result_hits", 0),
        "total_signal_score": metrics.get("total_signal_score", 0),
        "total_authority_score": metrics.get("total_authority_score", 0),
        "multi_result_support": metrics.get("multi_result_support", False),
        "weak_institution_evidence": metrics.get("weak_institution_evidence", False),
        "dominant_negative_signals": metrics.get("dominant_negative_signals", False),
        "passed_survey": actual_pass,
        "met_expectation_strict": strict_expected,
    }


def build_error_result(name: str, expected_group: str, exc: Exception) -> dict:
    return {
        "name": name,
        "expected_group": expected_group,
        "search_results": -1,
        "survey_results": -1,
        "bucket": "error",
        "score": None,
        "summary": str(exc),
        "reason_codes": "",
        "search_mode_used": "error",
        "enhanced_escalated": False,
        "rescue_attempted": False,
        "basic_result_count": -1,
        "final_result_count": -1,
        "elapsed_seconds": None,
        "escalation_reason": "",
        "rescue_reason": "",
        "enhanced_reason": "",
        "borderline_subtype": "",
        "backend_used": "",
        "cache_hit": False,
        "network_queries_used": None,
        "ddg_manual_retry_used": None,
        "ddg_browser_fallback_used": None,
        "bing_fallback_used": None,
        "network_attempt_count": None,
        "cache_hits_total": None,
        "cache_misses_total": None,
        "used_rescue_query": False,
        "stage": "",
        "max_relevance_score": None,
        "purdue_mentions": None,
        "purdue_domain_hits": None,
        "edu_hits": None,
        "explicit_connection_hits": None,
        "authoritative_institution_hits": None,
        "person_institution_hits": None,
        "academic_role_hits": None,
        "degree_hits": None,
        "joint_campus_hits": None,
        "negative_keyword_hits": None,
        "negative_result_hits": None,
        "total_signal_score": None,
        "total_authority_score": None,
        "multi_result_support": None,
        "weak_institution_evidence": None,
        "dominant_negative_signals": None,
        "passed_survey": False,
        "met_expectation_strict": False,
    }


async def run_validation(df: pd.DataFrame) -> pd.DataFrame:
    semaphore = asyncio.Semaphore(CONCURRENT_SEARCHES)
    total = len(df)

    async def worker(idx: int, row) -> dict:
        print(f"[{idx}/{total}] queued: {row.expected_group}: {row.name}")
        async with semaphore:
            print(f"[{idx}/{total}] running: {row.expected_group}: {row.name}")
            try:
                return await survey_name(row.name, row.expected_group)
            except Exception as exc:
                return build_error_result(row.name, row.expected_group, exc)

    tasks = [
        asyncio.create_task(worker(idx, row))
        for idx, row in enumerate(df.itertuples(index=False), start=1)
    ]
    rows = await asyncio.gather(*tasks)
    results_df = pd.DataFrame(rows)
    basic_only = int((results_df["search_mode_used"] == "basic_only").sum()) if not results_df.empty else 0
    escalated = int(results_df["enhanced_escalated"].fillna(False).astype(bool).sum()) if not results_df.empty else 0
    rescue_attempted = int(results_df["rescue_attempted"].fillna(False).astype(bool).sum()) if not results_df.empty else 0
    avg_elapsed = round(results_df["elapsed_seconds"].dropna().mean(), 2) if "elapsed_seconds" in results_df else None
    avg_network_queries = round(results_df["network_queries_used"].dropna().mean(), 2) if "network_queries_used" in results_df else None
    avg_network_attempts = round(results_df["network_attempt_count"].dropna().mean(), 2) if "network_attempt_count" in results_df else None
    cache_hit_rate = round(results_df["cache_hit"].fillna(False).astype(bool).mean(), 3) if "cache_hit" in results_df and not results_df.empty else None
    slow_ddg_fallback_names = int((results_df["ddg_manual_retry_used"].fillna(False).astype(bool) | results_df["ddg_browser_fallback_used"].fillna(False).astype(bool)).sum()) if not results_df.empty else 0
    browser_fallback_names = int(results_df["ddg_browser_fallback_used"].fillna(False).astype(bool).sum()) if not results_df.empty else 0
    bing_fallback_names = int(results_df["bing_fallback_used"].fillna(False).astype(bool).sum()) if not results_df.empty else 0
    borderline_negative_count = int(((results_df["expected_group"] == "super_non_related") & (results_df["bucket"] == "borderline")).sum()) if not results_df.empty else 0
    hard_no_negative_count = int(((results_df["expected_group"] == "super_non_related") & (results_df["bucket"] == "hard_no")).sum()) if not results_df.empty else 0
    avg_stages = round((1 + results_df["rescue_attempted"].fillna(False).astype(int) + results_df["enhanced_escalated"].fillna(False).astype(int)).mean(), 2) if not results_df.empty else None
    print({
        "policy": VALIDATION_POLICY,
        "basic_only_names": basic_only,
        "enhanced_escalations": escalated,
        "rescue_attempts": rescue_attempted,
        "average_elapsed_seconds": avg_elapsed,
        "average_network_queries": avg_network_queries,
        "average_network_attempts": avg_network_attempts,
        "cache_hit_rate": cache_hit_rate,
        "slow_ddg_fallback_names": slow_ddg_fallback_names,
        "ddg_browser_fallback_names": browser_fallback_names,
        "bing_fallback_names": bing_fallback_names,
        "negative_hard_no_count": hard_no_negative_count,
        "negative_borderline_count": borderline_negative_count,
        "average_search_stages": avg_stages,
    })
    return results_df


In [152]:
run_start = time.perf_counter()
results_df = await run_validation(cohort_df)
total_elapsed = round(time.perf_counter() - run_start, 2)
print({"total_elapsed_seconds": total_elapsed})
if SAVE_RESULTS:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved to {OUTPUT_PATH}")
results_df.head(10)


[1/50] queued: purdue_related: Neil Armstrong
[1/50] running: purdue_related: Neil Armstrong
[2/50] queued: purdue_related: Drew Brees
[2/50] running: purdue_related: Drew Brees
[3/50] queued: purdue_related: Amelia Earhart
[3/50] running: purdue_related: Amelia Earhart
[4/50] queued: purdue_related: Mitch Daniels
[4/50] running: purdue_related: Mitch Daniels
[5/50] queued: purdue_related: France A. Cordova
[5/50] running: purdue_related: France A. Cordova
[6/50] queued: purdue_related: Mung Chiang
[6/50] running: purdue_related: Mung Chiang
[7/50] queued: purdue_related: Eugene H. Spafford
[7/50] running: purdue_related: Eugene H. Spafford
[8/50] queued: purdue_related: Ei-ichi Negishi
[8/50] running: purdue_related: Ei-ichi Negishi
[9/50] queued: purdue_related: Herbert C. Brown
[9/50] running: purdue_related: Herbert C. Brown
[10/50] queued: purdue_related: R. Graham Cooks
[10/50] running: purdue_related: R. Graham Cooks
[11/50] queued: purdue_related: John B. Fenn
[12/50] queued: p

,name,expected_group,search_results,survey_results,search_mode_used,enhanced_escalated,rescue_attempted,basic_result_count,final_result_count,elapsed_seconds,...,joint_campus_hits,negative_keyword_hits,negative_result_hits,total_signal_score,total_authority_score,multi_result_support,weak_institution_evidence,dominant_negative_signals,passed_survey,met_expectation_strict
0,Neil Armstrong,purdue_related,12,12,basic_only_demoted,False,False,12,12,3.93,...,0,3,0,420,23,False,False,False,False,False
1,Drew Brees,purdue_related,12,24,basic_plus_enhanced,True,True,12,24,31.23,...,0,4,0,150,65,False,False,False,True,True
2,Amelia Earhart,purdue_related,12,24,basic_plus_enhanced,True,True,12,24,34.26,...,0,2,0,180,73,False,False,False,True,True
3,Mitch Daniels,purdue_related,12,12,basic_only_demoted,False,False,12,12,3.73,...,0,0,0,140,5,False,False,False,False,False
4,France A. Cordova,purdue_related,12,26,basic_plus_enhanced,True,True,12,26,30.32,...,0,1,0,320,70,True,False,False,True,True
5,Mung Chiang,purdue_related,12,12,basic_only,False,False,12,12,9.17,...,0,1,0,160,8,True,False,False,True,True
6,Eugene H. Spafford,purdue_related,12,12,basic_only,False,False,12,12,6.00,...,0,1,0,330,45,True,False,False,True,True
7,Ei-ichi Negishi,purdue_related,12,12,basic_only,False,False,12,12,4.30,...,0,1,0,410,10,True,False,False,True,True
8,Herbert C. Brown,purdue_related,12,17,basic_only,False,True,12,17,17.24,...,0,1,0,520,40,True,False,False,True,True
9,R. Graham Cooks,purdue_related,12,12,basic_only,False,False,12,12,12.13,...,0,0,0,310,15,True,False,False,True,True


## Summary View

In [153]:
summary_df = (
    results_df.groupby("expected_group")
    .agg(
        names=("name", "count"),
        hard_no=("bucket", lambda s: int((s == "hard_no").sum())),
        borderline=("bucket", lambda s: int((s == "borderline").sum())),
        plausible=("bucket", lambda s: int((s == "plausible").sum())),
        errors=("bucket", lambda s: int((s == "error").sum())),
        met_expectation_strict=("met_expectation_strict", "sum"),
    )
    .reset_index()
)
summary_df["strict_accuracy"] = (summary_df["met_expectation_strict"] / summary_df["names"]).round(3)
summary_df


,expected_group,names,hard_no,borderline,plausible,errors,met_expectation_strict,strict_accuracy
0,purdue_related,25,2,5,18,0,23,0.92
1,super_non_related,25,19,5,1,0,19,0.76


In [154]:
bucket_pivot = pd.crosstab(results_df["expected_group"], results_df["bucket"], margins=True)
bucket_pivot


bucket,borderline,hard_no,plausible,All
expected_group,,,,
purdue_related,5,2,18,25
super_non_related,5,19,1,25
All,10,21,19,50


In [155]:
display_cols = [
    "expected_group",
    "name",
    "bucket",
    "score",
    "used_rescue_query",
    "max_relevance_score",
    "purdue_domain_hits",
    "explicit_connection_hits",
    "authoritative_institution_hits",
    "person_institution_hits",
    "negative_result_hits",
    "summary",
    "reason_codes",
    "met_expectation_strict",
]
results_df.sort_values(["expected_group", "bucket", "score"], ascending=[True, True, False])[display_cols]


,expected_group,name,bucket,score,used_rescue_query,max_relevance_score,purdue_domain_hits,explicit_connection_hits,authoritative_institution_hits,person_institution_hits,negative_result_hits,summary,reason_codes,met_expectation_strict
1,purdue_related,Drew Brees,borderline,60,True,24,13,1,0,19,0,Borderline evidence: Explicit connection langu...,explicit_connection|institution_domain_hit|edu...,True
17,purdue_related,Samuel D. Allen,borderline,60,True,35,4,1,0,6,0,Borderline evidence: Explicit connection langu...,explicit_connection|institution_domain_hit|edu...,True
2,purdue_related,Amelia Earhart,borderline,59,True,28,14,1,0,21,0,Borderline evidence: Explicit connection langu...,explicit_connection|institution_domain_hit|edu...,True
24,purdue_related,Jozef L. Kokini,borderline,59,True,26,12,1,0,15,0,Borderline evidence: Explicit connection langu...,explicit_connection|institution_domain_hit|edu...,True
21,purdue_related,Marisol Nunez,borderline,50,True,16,5,0,4,0,0,Borderline evidence: Authoritative institution...,authoritative_institution_page|institution_dom...,True
0,purdue_related,Neil Armstrong,hard_no,38,False,31,5,0,0,11,0,Borderline evidence: Institution domain hit fo...,institution_domain_hit|edu_signal|person_insti...,False
3,purdue_related,Mitch Daniels,hard_no,29,False,21,1,0,0,12,0,Borderline evidence: Institution domain hit fo...,institution_domain_hit|edu_signal|person_insti...,False
6,purdue_related,Eugene H. Spafford,plausible,83,False,25,4,5,2,11,0,Explicit connection language found in search r...,explicit_connection|authoritative_institution_...,True
23,purdue_related,Timothy D. Sands,plausible,81,False,21,3,2,2,10,0,Explicit connection language found in search r...,explicit_connection|authoritative_institution_...,True
18,purdue_related,Edward M. Purcell,plausible,78,False,29,3,3,1,9,0,Explicit connection language found in search r...,explicit_connection|authoritative_institution_...,True


## Failure Inspection

These are the cases worth reading first:
- Purdue-related names that incorrectly landed in `hard_no`
- super non-related names that escaped `hard_no`


In [156]:
purdue_false_negatives = results_df[
    (results_df["expected_group"] == "purdue_related")
    & (results_df["bucket"] == "hard_no")
][display_cols]
purdue_false_negatives


,expected_group,name,bucket,score,used_rescue_query,max_relevance_score,purdue_domain_hits,explicit_connection_hits,authoritative_institution_hits,person_institution_hits,negative_result_hits,summary,reason_codes,met_expectation_strict
0,purdue_related,Neil Armstrong,hard_no,38,False,31,5,0,0,11,0,Borderline evidence: Institution domain hit fo...,institution_domain_hit|edu_signal|person_insti...,False
3,purdue_related,Mitch Daniels,hard_no,29,False,21,1,0,0,12,0,Borderline evidence: Institution domain hit fo...,institution_domain_hit|edu_signal|person_insti...,False


In [157]:
non_purdue_escapes = results_df[
    (results_df["expected_group"] == "super_non_related")
    & (results_df["bucket"] != "hard_no")
][display_cols]
non_purdue_escapes


,expected_group,name,bucket,score,used_rescue_query,max_relevance_score,purdue_domain_hits,explicit_connection_hits,authoritative_institution_hits,person_institution_hits,negative_result_hits,summary,reason_codes,met_expectation_strict
38,super_non_related,Tom Hanks,borderline,60,True,33,5,1,0,3,0,Borderline evidence: Explicit connection langu...,explicit_connection|institution_domain_hit|edu...,False
39,super_non_related,Oprah Winfrey,borderline,45,True,31,5,0,1,5,0,Borderline evidence: Authoritative institution...,authoritative_institution_page|institution_dom...,False
41,super_non_related,Zendaya,borderline,21,False,18,1,0,0,0,0,Borderline evidence: Institution domain hit fo...,institution_domain_hit|edu_signal|strong_relev...,False
43,super_non_related,Shakira,plausible,84,True,29,14,2,4,4,0,Explicit connection language found in search r...,explicit_connection|authoritative_institution_...,False
45,super_non_related,Robert Downey Jr,borderline,28,False,13,1,0,0,0,0,Borderline evidence: Institution domain hit fo...,institution_domain_hit|edu_signal|strong_relev...,False
47,super_non_related,Stephen Curry,borderline,45,True,22,2,0,1,3,0,Borderline evidence: Authoritative institution...,authoritative_institution_page|institution_dom...,False


In [158]:
metric_cols = [
    "score",
    "max_relevance_score",
    "purdue_mentions",
    "purdue_domain_hits",
    "edu_hits",
    "explicit_connection_hits",
    "authoritative_institution_hits",
    "person_institution_hits",
    "academic_role_hits",
    "degree_hits",
    "negative_result_hits",
    "total_signal_score",
    "total_authority_score",
]
results_df.groupby("expected_group")[metric_cols].mean(numeric_only=True).round(2)


,score,max_relevance_score,purdue_mentions,purdue_domain_hits,edu_hits,explicit_connection_hits,authoritative_institution_hits,person_institution_hits,academic_role_hits,degree_hits,negative_result_hits,total_signal_score,total_authority_score
expected_group,,,,,,,,,,,,,
purdue_related,64.12,26.52,13.48,4.88,5.72,2.32,0.64,11.36,2.00,0.84,0.04,274.0,33.16
super_non_related,31.80,19.36,5.36,1.92,2.84,0.12,0.24,1.60,0.12,0.12,0.00,113.6,17.16


## Cleanup

In [159]:
await close_search_clients()
await close_session()
print("Closed search and LLM sessions.")


Closed search and LLM sessions.
